In [ ]:
import os
import pandas as pd
from getpass import getpass

from dbrepo.RestClient import RestClient

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username=" ",
    password=getpass("DBRepo password: ")
)

DB_ID = " "

db = client.get_database(DB_ID)

print("Database:", db.name)
print("Tables:", len(db.tables))
print("Views:", len(db.views))
print("Accesses:", db.accesses)

for table in db.tables:
    print("-", table.name)

Database: data_stew_grp22_air_quality
Tables: 3
Views: 3
Accesses: [DatabaseAccess(type=<AccessType.WRITE_ALL: 'write_all'>, user=UserBrief(username='12450741', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None))]
- t_measurement
- t_time
- t_station


In [ ]:
TABLE_IDS = {
    "t_measurement": "",
    "t_time": "",
    "t_station": "",
}

COLUMN_IDS = {
    ("t_station", "latitude"): "",
    ("t_station", "longitude"): "",

    ("t_measurement", "SO2"): "",
    ("t_measurement", "NO"): "",
    ("t_measurement", "NO2"): "",
    ("t_measurement", "CO"): "",
    ("t_measurement", "PM10"): "",
    ("t_measurement", "O3"): "",
    ("t_measurement", "PM25"): "",

    ("t_measurement", "wind_direction"): "",
    ("t_measurement", "wind_speed"): "",
    ("t_measurement", "temperature"): "",
    ("t_measurement", "humidity"): "",
    ("t_measurement", "pressure"): "",
    ("t_measurement", "solar_radiation"): "",
    ("t_measurement", "rain"): "",

    ("t_measurement", "BEN"): "",
    ("t_measurement", "TOL"): "",
    ("t_measurement", "MXIL"): "",
}

In [3]:
unit_mappings = [
    # Coordinates
    ("t_station", "latitude", "degree", "http://www.ontology-of-units-of-measure.org/resource/om-2/degree"),
    ("t_station", "longitude", "degree", "http://www.ontology-of-units-of-measure.org/resource/om-2/degree"),

    # Pollutants / chemical concentrations
    ("t_measurement", "SO2", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "NO", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "NO2", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "CO", "milligram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/milligramPerCubicMetre"),
    ("t_measurement", "PM10", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "PM25", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "O3", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "BEN", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "TOL", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),
    ("t_measurement", "MXIL", "microgram per cubic metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/microgramPerCubicMetre"),

    # Meteorological attributes
    ("t_measurement", "wind_direction", "degree", "http://www.ontology-of-units-of-measure.org/resource/om-2/degree"),
    ("t_measurement", "wind_speed", "metre per second", "http://www.ontology-of-units-of-measure.org/resource/om-2/metrePerSecond-Time"),
    ("t_measurement", "temperature", "degree Celsius", "http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius"),
    ("t_measurement", "humidity", "percent", "http://www.ontology-of-units-of-measure.org/resource/om-2/percent"),
    ("t_measurement", "pressure", "hectopascal", "http://www.ontology-of-units-of-measure.org/resource/om-2/hectopascal"),
    ("t_measurement", "solar_radiation", "watt per square metre", "http://www.ontology-of-units-of-measure.org/resource/om-2/wattPerSquareMetre"),
    ("t_measurement", "rain", "millimetre", "http://www.ontology-of-units-of-measure.org/resource/om-2/millimetre"),
]

unit_mapping_df = pd.DataFrame(
    unit_mappings,
    columns=["table", "column", "unit_label", "expected_unit_uri"]
)

unit_mapping_df

,table,column,unit_label,expected_unit_uri
0,t_station,latitude,degree,http://www.ontology-of-units-of-measure.org/re...
1,t_station,longitude,degree,http://www.ontology-of-units-of-measure.org/re...
2,t_measurement,SO2,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...
3,t_measurement,NO,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...
4,t_measurement,NO2,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...
5,t_measurement,CO,milligram per cubic metre,http://www.ontology-of-units-of-measure.org/re...
6,t_measurement,PM10,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...
7,t_measurement,PM25,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...
8,t_measurement,O3,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...
9,t_measurement,BEN,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...


In [4]:
def get_column_metadata(table_name, column_name):
    table_id = TABLE_IDS[table_name]
    column_id = COLUMN_IDS[(table_name, column_name)]

    table_obj = client.get_table(DB_ID, table_id)

    col_obj = next(
        c for c in table_obj.columns
        if c.id == column_id
    )

    return {
        "table": table_name,
        "column": column_name,
        "concept_uri": getattr(col_obj, "concept_uri", None),
        "unit_uri": getattr(col_obj, "unit_uri", None)
    }

In [ ]:
before_rows = []

for _, mapping in unit_mapping_df.iterrows():
    meta = get_column_metadata(
        mapping["table"],
        mapping["column"]
    )

    before_rows.append({
        "table": mapping["table"],
        "column": mapping["column"],
        "unit_label": mapping["unit_label"],
        "expected_unit_uri": mapping["expected_unit_uri"],
        "current_unit_uri": meta["unit_uri"],
        "already_updated": meta["unit_uri"] == mapping["expected_unit_uri"]
    })

unit_check_before_df = pd.DataFrame(before_rows)

unit_check_before_df

In [6]:
unit_upload_results = []

for _, row in unit_check_before_df.iterrows():

    table_name = row["table"]
    column_name = row["column"]
    expected_unit_uri = row["expected_unit_uri"]

    if row["already_updated"] == True:
        unit_upload_results.append({
            "table": table_name,
            "column": column_name,
            "unit_label": row["unit_label"],
            "unit_uri": expected_unit_uri,
            "status": "already_updated",
            "error": None
        })
        continue

    try:
        table_id = TABLE_IDS[table_name]
        column_id = COLUMN_IDS[(table_name, column_name)]

        # Preserve existing concept URI while adding/updating unit URI
        current_meta = get_column_metadata(table_name, column_name)
        existing_concept_uri = current_meta["concept_uri"]

        client.update_table_column(
            database_id=DB_ID,
            table_id=table_id,
            column_id=column_id,
            concept_uri=existing_concept_uri,
            unit_uri=expected_unit_uri
        )

        unit_upload_results.append({
            "table": table_name,
            "column": column_name,
            "unit_label": row["unit_label"],
            "unit_uri": expected_unit_uri,
            "status": "success",
            "error": None
        })

    except Exception as e:
        unit_upload_results.append({
            "table": table_name,
            "column": column_name,
            "unit_label": row["unit_label"],
            "unit_uri": expected_unit_uri,
            "status": "failed",
            "error": repr(e)
        })

unit_upload_df = pd.DataFrame(unit_upload_results)

unit_upload_df

,table,column,unit_label,unit_uri,status,error
0,t_station,latitude,degree,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
1,t_station,longitude,degree,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
2,t_measurement,SO2,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
3,t_measurement,NO,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
4,t_measurement,NO2,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
5,t_measurement,CO,milligram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
6,t_measurement,PM10,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
7,t_measurement,PM25,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
8,t_measurement,O3,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None
9,t_measurement,BEN,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,already_updated,None


In [8]:
after_rows = []

for _, mapping in unit_mapping_df.iterrows():
    meta = get_column_metadata(
        mapping["table"],
        mapping["column"]
    )

    after_rows.append({
        "table": mapping["table"],
        "column": mapping["column"],
        "unit_label": mapping["unit_label"],
        "expected_unit_uri": mapping["expected_unit_uri"],
        "current_unit_uri": meta["unit_uri"],
        "updated_correctly": meta["unit_uri"] == mapping["expected_unit_uri"]
    })

unit_check_after_df = pd.DataFrame(after_rows)

unit_check_after_df

,table,column,unit_label,expected_unit_uri,current_unit_uri,updated_correctly
0,t_station,latitude,degree,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
1,t_station,longitude,degree,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
2,t_measurement,SO2,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
3,t_measurement,NO,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
4,t_measurement,NO2,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
5,t_measurement,CO,milligram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
6,t_measurement,PM10,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
7,t_measurement,PM25,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
8,t_measurement,O3,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
9,t_measurement,BEN,microgram per cubic metre,http://www.ontology-of-units-of-measure.org/re...,http://www.ontology-of-units-of-measure.org/re...,True
